# AI工学101 — 第30回

## 学習曲線・検証曲線：モデルが「なぜ」うまくいかないのかを診断する

今日はかなり大事な回だぞ。

第20回で、

```text
Underfitting
Overfitting
```

を学び、第29回では、

```text
Hyperparameter Search
```

までやりました。

でも、ここで一つ問題があります。

> **「性能が悪い」という結果を見ただけでは、何を直せばいいのか分からない。**

例えば、

```text
Accuracy = 82%
```

だったとしても、

* データが少ない？
* モデルが単純すぎる？
* モデルが複雑すぎる？
* ノイズが多い？
* 特徴量が悪い？
* そもそも問題が難しい？

が分かりません。

そこで今日は、

> **モデルの失敗原因をグラフから診断する**

方法を学びます。

---

# 🎯 今日のゴール

今日は、

* Bias / Varianceの直感を理解する
* Underfitting / Overfittingを診断できる
* Learning Curveを使える
* Validation Curveを使える
* 「データを増やすべきか」「モデルを複雑にすべきか」を考えられる
* ハイパーパラメータ探索の前に診断する重要性を理解する

ところまで進みます。

---

# 📖 講義：約20分

## 1. BiasとVariance

まず大枠。

モデルの誤差を考えるとき、

```text
Bias
Variance
Noise
```

という3つの要素を考えることがあります。

かなり直感的に言うと、

### High Bias

```text
モデルが単純すぎる
↓
データの構造を表現できない
↓
Underfitting
```

---

### High Variance

```text
モデルがデータに敏感すぎる
↓
trainデータに合わせすぎる
↓
Overfitting
```

---

## 2. グラフで考える

例えばモデルの複雑さを横軸にすると、

```text
誤差
 ↑
 │\
 │ \
 │  \       Test Error
 │   \     /\
 │    \   /  \
 │     \_/    \
 │
 │──────────────→ モデルの複雑さ
```

ざっくり、

```text
単純すぎる
↓
Biasが大きい

複雑すぎる
↓
Varianceが大きい
```

という構造があります。

---

# 🧠 3. でも「もっと複雑にすればいい」は危険

例えば、

```text
train accuracy = 99%
test accuracy = 75%
```

だったとします。

これは、

> **モデルをさらに複雑にすればいい**

ではありません。

むしろ、

```text
すでにtrainに適合しすぎている
↓
Overfitting
```

かもしれません。

逆に、

```text
train accuracy = 75%
test accuracy = 74%
```

なら、

```text
train自体でうまく説明できていない
```

ので、

**Underfitting**

の可能性があります。

この違いを診断するのが今日のテーマです。

---

# 💻 実習1：データを用意する

今回は少し大きめの分類データを作ります。

```python
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    random_state=42
)
```

分割。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

# 💻 実習2：まず普通にRandom Forest

```python
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)
```

評価。

```python
print(
    "train:",
    model.score(
        X_train,
        y_train
    )
)

print(
    "test:",
    model.score(
        X_test,
        y_test
    )
)
```

まず、

```text
train
test
```

の差を確認します。

---

# 📖 4. Learning Curve

ここから今日の主役。

**Learning Curve** は、

> **学習に使うデータ量を増やしたとき、train性能とvalidation性能がどう変化するか**

を見るグラフです。

つまり、

```text
データ10%
↓
データ20%
↓
データ40%
↓
データ60%
↓
データ80%
↓
データ100%
```

と学習データを増やして、

```text
train score
validation score
```

を観察します。

---

# 💻 実習3：Learning Curve

```python
from sklearn.model_selection import learning_curve
```

計算。

```python
train_sizes, train_scores, val_scores = learning_curve(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    train_sizes=[
        0.1,
        0.25,
        0.5,
        0.75,
        1.0
    ],
    n_jobs=-1
)
```

平均を取ります。

```python
train_mean = train_scores.mean(
    axis=1
)

val_mean = val_scores.mean(
    axis=1
)
```

---

# 💻 実習4：Learning Curveを描く

```python
import matplotlib.pyplot as plt

plt.plot(
    train_sizes,
    train_mean,
    marker="o",
    label="train"
)

plt.plot(
    train_sizes,
    val_mean,
    marker="o",
    label="validation"
)

plt.xlabel(
    "Training examples"
)

plt.ylabel(
    "Accuracy"
)

plt.legend()

plt.show()
```

---

# 👀 何を見るのか？

Learning Curveでは、

```text
train score
validation score
```

の2本の線を見ます。

---

# 🧠 ケースA：Underfitting

例えば、

```text
train     ───────── 80%
validation ──────── 78%
```

のように、

**両方とも低く、差も小さい**

場合。

これは、

```text
モデルの表現力不足
```

の可能性があります。

対策候補：

```text
モデルを複雑にする
特徴量を改善する
正則化を弱める
```

など。

---

# 🧠 ケースB：Overfitting

例えば、

```text
train       ───────── 99%
validation  ──────── 78%
```

のように、

**trainが高く、validationが低い**

場合。

これは、

```text
High Variance
↓
Overfitting
```

の可能性があります。

対策候補：

```text
モデルを単純化
正則化
データを増やす
特徴量を減らす
```

など。

---

# 🧠 ケースC：データを増やすと改善しそう

ここがLearning Curveの面白いところ。

例えば、

```text
train      98 → 94 → 90
validation 70 → 78 → 85
```

のように、

**データ量を増やすほどvalidation性能が上がっている**

場合、

> **もっとデータを集める価値がある**

可能性があります。

つまりLearning Curveは、

> **「モデルをいじる前に、データを増やすべきか？」**

を考える材料にもなります。

---

# 💻 実習5：Training Sizeを確認

```python
print(
    train_sizes
)
```

```python
print(
    train_mean
)

print(
    val_mean
)
```

数値でも確認しましょう。

グラフだけでなく、

```text
データ量
train
validation
```

を表として見るのも良いです。

---

# 📖 5. Validation Curve

次は、

**Validation Curve**

です。

Learning Curveが、

```text
データ量
```

を変えたのに対して、

Validation Curveは、

> **あるハイパーパラメータを変えたとき、train/validation性能がどう変化するか**

を見ます。

---

# 💻 実習6：max_depthを変える

Random Forestについて、

```text
max_depth
```

を変えてみます。

```python
from sklearn.model_selection import validation_curve
```

計算。

```python
param_range = [
    1,
    2,
    3,
    5,
    10,
    20
]
```

```python
train_scores, val_scores = validation_curve(
    model,
    X,
    y,
    param_name="max_depth",
    param_range=param_range,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
```

平均。

```python
train_mean = train_scores.mean(
    axis=1
)

val_mean = val_scores.mean(
    axis=1
)
```

---

# 💻 実習7：Validation Curveを描く

```python
plt.plot(
    param_range,
    train_mean,
    marker="o",
    label="train"
)

plt.plot(
    param_range,
    val_mean,
    marker="o",
    label="validation"
)

plt.xlabel(
    "max_depth"
)

plt.ylabel(
    "Accuracy"
)

plt.legend()

plt.show()
```

---

# 👀 何が分かる？

例えば、

```text
max_depth = 1
```

では、

```text
train 低い
validation 低い
```

かもしれない。

これは、

```text
Underfitting
```

の可能性。

---

そして、

```text
max_depth = 5
```

で、

```text
train 高い
validation 高い
```

になったら、

**適切な複雑さ**

かもしれません。

---

さらに、

```text
max_depth = 20
```

で、

```text
train = 100%
validation = 75%
```

になったら、

**Overfitting**

です。

---

# 🧠 6. Validation Curveは「最適領域」を探す

つまり、

```text
モデルが単純すぎる
        ↓
性能低い

       ↓

ちょうどいい複雑さ
        ↓
性能高い

       ↓

複雑すぎる
        ↓
過学習
```

という構造を見ることができます。

---

# 💻 実習8：learning_rateでもやる

第23回のGradient Boostingに戻ります。

```python
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    random_state=42
)
```

```python
param_range = [
    0.01,
    0.03,
    0.05,
    0.1,
    0.2,
    0.3
]
```

```python
train_scores, val_scores = validation_curve(
    gb,
    X,
    y,
    param_name="learning_rate",
    param_range=param_range,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
```

平均。

```python
train_mean = train_scores.mean(
    axis=1
)

val_mean = val_scores.mean(
    axis=1
)
```

描画。

```python
plt.plot(
    param_range,
    train_mean,
    marker="o",
    label="train"
)

plt.plot(
    param_range,
    val_mean,
    marker="o",
    label="validation"
)

plt.xlabel(
    "learning_rate"
)

plt.ylabel(
    "Accuracy"
)

plt.legend()

plt.show()
```

---

# 🔥 第29回との接続

ここが今日かなり重要。

第29回では、

```text
GridSearchCV
```

で、

> 「どのパラメータが良い？」

を探しました。

今日はその前段階として、

> **「そもそも、このパラメータを動かすとモデルがどう振る舞う？」**

を見ています。

つまり、

```text
Validation Curve
      ↓
モデルの挙動を理解
      ↓
探索範囲を設計
      ↓
GridSearch / RandomizedSearch
```

という流れが作れます。

これはかなり良い実験設計です。

---

# 🧠 7. 「データを増やす」vs「モデルを変える」

Learning Curveから得られる重要な判断があります。

例えば、

### パターン1

```text
train ─────── 95%
validation ── 75%
```

でデータを増やしても、

```text
validation
75 → 76 → 76
```

とほぼ伸びない。

この場合、

> **単純にデータを増やすだけでは大きく改善しない**

可能性があります。

---

### パターン2

```text
validation
70 → 75 → 80 → 85
```

と増えていく。

これは、

> **データを増やす価値がありそう**

です。

---

# 🧠 8. AI開発では「もっとデータ！」だけではない

性能が悪いと、

```text
データを増やそう！
```

となりがちです。

でも、

```text
特徴量が悪い
↓
データを増やしても限界
```

かもしれない。

逆に、

```text
モデルが単純すぎる
```

なら、

```text
モデルを強くする
```

ほうが効く。

つまり、

> **性能改善には診断が必要。**

これが今日の核心です。

---

# ✍️ 演習

## 問1

次の結果を見てください。

```text
train accuracy = 0.78
validation accuracy = 0.77
```

このモデルは、

```text
Underfitting
Overfitting
```

のどちらが疑わしいでしょうか？

---

## 問2

次はどうでしょう？

```text
train accuracy = 0.99
validation accuracy = 0.76
```

---

## 問3

Learning Curveで、

```text
training dataを増やす
↓
validation scoreが継続的に上昇
```

していたら、

> **何を試す価値があるでしょうか？**

---

## 問4

Validation Curveで、

```text
max_depth = 1
→ train低い / validation低い

max_depth = 5
→ train高い / validation高い

max_depth = 20
→ train非常に高い / validation低下
```

となった場合、

> **どの領域が候補になりそうでしょうか？**

---

# 👾 ボス戦：モデル診断レポート

Random Forestについて、

```text
max_depth
```

のValidation Curveを作り、

```text
1
2
3
5
10
20
```

を比較してください。

さらにLearning Curveを作り、

```text
training data size
```

と

```text
train score
validation score
```

を確認します。

そして最後に、

> **「このモデルを改善するとしたら、まず何を試すか？」**

を決めてください。

候補は、

```text
A. データを増やす
B. モデルを複雑にする
C. モデルを単純にする
D. 特徴量を改善する
E. 正則化を調整する
```

です。

ただし、**理由をグラフから説明する**こと。

---

# 🌱 今日のまとめ

今日の核心は、

> **モデルの性能を上げる前に、なぜ性能が悪いのかを診断する。**

です。

これまで、

```text
GridSearch
```

で、

> 「良いパラメータを探す」

ことをしていました。

今日は、

```text
Learning Curve
Validation Curve
```

で、

> **「モデルがどういう状態なのかを見る」**

ことを学びました。

---

## 2つの曲線を覚えよう

### Learning Curve

```text
横軸：
学習データ量

↓

「もっとデータが必要？」
```

---

### Validation Curve

```text
横軸：
ハイパーパラメータ

↓

「モデルの複雑さは適切？」
```

---

# 🧭 AI工学101・現在地

ここまで来ると、

```text
データ
 ↓
前処理
 ↓
特徴量
 ↓
モデル
 ↓
学習
 ↓
評価
 ↓
診断
 ↓
改善
 ↓
再評価
```

という**機械学習開発の反復ループ**が完成しつつあります。

そして、

```text
「Accuracyが低い」
```

から、

```text
「Underfittingっぽい」
↓
「モデル容量を増やしてみよう」

あるいは

「Overfittingっぽい」
↓
「正則化・データ・特徴量を検討しよう」
```

と、**原因→対策**へ移れるようになってきました。

これが「基礎体力」のかなり重要な部分です。

---

# 🔜 第31回

## データリークと再現性：機械学習実験を「正しく」する

次回は、今まで何度も出てきた、

> **「testデータを見ちゃダメ」**

を、もっと深く掘ります。

扱うのは、

* Data Leakage
* Target Leakage
* Train/Test contamination
* Pipelineによるリーク防止
* `random_state`
* 実験の再現性
* CVとtest setの正しい役割
* 「精度99%なのに本番では使えない」タイプの事故

です。

ここは地味だけど、**AI開発の工学的な基礎としてめちゃくちゃ重要な回**になります。
